## **Análisis componentes para crear el nodo worker `worker`**

### **`Dockerfile`**

**`FROM ubuntu:latest`**: Esta línea indica que la imagen que se va a construir se basará en la imagen base de Ubuntu, específicamente la versión "latest" (última). En Docker, una imagen base proporciona el sistema de archivos inicial y los binarios necesarios para ejecutar una aplicación. En este caso, la imagen base es Ubuntu, lo que significa que la imagen de Docker contendrá un sistema de archivos similar al de Ubuntu.

**`MAINTAINER Alfonso Perez Lino`**: Esta línea se utiliza para especificar el nombre del mantenedor o creador de la imagen. En versiones anteriores de Docker, esta línea era importante para proporcionar información sobre quién mantenía o creaba la imagen. Sin embargo, a partir de Docker 1.13, la instrucción **MAINTAINER** está obsoleta en favor de las etiquetas **LABEL**. Ahora, la información del mantenedor se suele agregar como una etiqueta de metadatos en lugar de utilizar la instrucción **MAINTAINER**.

**`LABEL creador="Alfonso Perez Lino"`**: Esta línea agrega metadatos a la imagen resultante. Los metadatos proporcionan información adicional sobre la imagen, como el nombre del creador, la versión, la descripción, etc. En este caso, se está utilizando la etiqueta creador para indicar el nombre del creador de la imagen. Esta información puede ser útil para otros usuarios que trabajen con la imagen, ya que proporciona contexto sobre su origen y su propósito.

In [ ]:
FROM hadoop-cluster-base
LABEL Alfonso Perez Lino

___

**`USER root`**: Esta línea indica que se cambiará al usuario root dentro del entorno de la imagen Docker. El usuario root es el superusuario en sistemas basados en Unix y tiene privilegios completos para realizar cualquier acción en el sistema. Al cambiar al usuario root, se obtienen permisos y privilegios completos dentro del entorno de la imagen, lo que permite ejecutar comandos que pueden requerir permisos elevados, como instalar paquetes, configurar el sistema, y realizar otras tareas administrativas.

In [ ]:
# Cambiamos al usuario root
USER root

___

**`WORKDIR ${BASE_DIR}`**: Este comando establece el directorio de trabajo actual dentro del contenedor. **WORKDIR** se utiliza para cambiar el directorio de trabajo en el contenedor para el resto de los comandos en el Dockerfile. En este caso, se establece el directorio de trabajo como el valor de la variable de entorno **BASE_DIR**, que es **/opt/bd**. Esto significa que todos los comandos posteriores en el Dockerfile se ejecutarán en el contexto de este directorio dentro del contenedor. Esto puede ser útil para organizar el espacio de trabajo y las operaciones del contenedor.

In [ ]:
WORKDIR ${BASE_DIR}

___

`echo "$LOG_TAG Crear directorio datanode"`: Este comando simplemente imprime un mensaje en la salida estándar que indica que se está creando el directorio del nodo worker (datanode). `$LOG_TAG` es una variable de entorno que probablemente contenga una etiqueta o identificador para el registro, como un nivel de gravedad o un nombre de aplicación.

`mkdir -p ${BASE_DIR}/hdfs/datanode`: Este comando crea un directorio llamado `datanode` dentro de la ruta especificada por `${BASE_DIR}/hdfs/`. La opción `-p` le dice al comando `mkdir` que cree todos los directorios necesarios en la ruta proporcionada si alguno de ellos aún no existe. `${BASE_DIR}` es una variable de entorno que apunta al directorio base **/opt/bd** donde se almacenarán los datos relacionados con Hadoop en el contenedor.

In [ ]:
#Creamos el directorio para el datanode
RUN echo "$LOG_TAG Crear directorio datanode" && \
    mkdir -p ${BASE_DIR}/hdfs/datanode 

___

Este fragmento de Dockerfile está copiando archivos de configuración y scripts desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd/hadoop/etc/hadoop` dentro del contenedor.

Aquí está lo que hace cada línea:

`COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml`: Copia el archivo `core-site.xml` desde el directorio `Config-files` de tu sistema local al directorio de configuración de Hadoop dentro del contenedor (`${HADOOP_CONF_DIR}/core-site.xml`).

`COPY Config-files/hdfs-site.xml ${HADOOP_CONF_DIR}/hdfs-site.xml`: Copia el archivo `hdfs-site.xml` de manera similar al anterior.

`COPY Config-files/yarn-site.xml ${HADOOP_CONF_DIR}/yarn-site.xml`: Copia el archivo `yarn-site.xml` de manera similar al anterior.

`COPY Config-files/mapred-site.xml ${HADOOP_CONF_DIR}/mapred-site.xml`: Copia el archivo `mapred-site.xml` de manera similar al anterior.

Estos comandos garantizan que los archivos de configuración necesarios para Hadoop se encuentren en el lugar correcto dentro del contenedor para que Hadoop pueda utilizarlos.

In [ ]:
#Copiamos los archivos de configuración que tenemos en el directorio "Config-files" de nuestro PC al 
#directorio /opt/bd/hadoop/etc/hadoop del contenedor.
RUN echo "$LOG_TAG Copiando archivos de configuración y script de inicio"
COPY Config-files/core-site.xml ${HADOOP_CONF_DIR}/core-site.xml
COPY Config-files/hdfs-site.xml ${HADOOP_CONF_DIR}/hdfs-site.xml
COPY Config-files/yarn-site.xml ${HADOOP_CONF_DIR}/yarn-site.xml
COPY Config-files/mapred-site.xml ${HADOOP_CONF_DIR}/mapred-site.xml

___

Este fragmento de Dockerfile está copiando dos scripts desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd` dentro del contenedor.

Aquí está lo que hace cada línea:

1. `COPY Config-files/start-daemons.sh ${BASE_DIR}/start-daemons.sh`: Copia el script `start-daemons.sh` desde el directorio `Config-files` de tu sistema local al directorio `/opt/bd` dentro del contenedor con el nombre `start-daemons.sh`.

2. `COPY Config-files/start-services.sh ${BASE_DIR}/start-services.sh`: Copia el script `start-services.sh` de manera similar al anterior, con el nombre `start-services.sh`.

Estos comandos garantizan que los scripts necesarios para iniciar los daemons de Hadoop y el servicio SSH se encuentren en el lugar correcto dentro del contenedor.

In [ ]:
# Script de inicio de daemons y SSH
COPY Config-files/start-daemons.sh ${BASE_DIR}/start-daemons.sh
COPY Config-files/start-services.sh ${BASE_DIR}/start-services.sh

___

Este fragmento de Dockerfile establece los permisos de ejecución para los scripts `start-daemons.sh` y `start-services.sh` dentro del contenedor. 

Aquí está lo que hace cada línea:

1. `chmod +x ${BASE_DIR}/start-daemons.sh`: Otorga permisos de ejecución al script `start-daemons.sh` ubicado en el directorio `${BASE_DIR}` dentro del contenedor.

2. `chmod +x ${BASE_DIR}/start-services.sh`: Similar al anterior, esta línea otorga permisos de ejecución al script `start-services.sh` ubicado en el directorio `${BASE_DIR}` dentro del contenedor. 

Esto asegura que los scripts sean ejecutables dentro del contenedor.

In [ ]:
# Establecemos los permisos
RUN echo "$LOG_TAG Cambiar permisos de ejecución" && \
    chmod +x ${BASE_DIR}/start-daemons.sh && \
    chmod +x ${BASE_DIR}/start-services.sh

___

Este fragmento de Dockerfile establece el directorio de trabajo como el directorio raíz de Hadoop especificado por la variable de entorno `${HADOOP_HOME}`. Luego, se ejecuta el script `start-services.sh` como el comando predeterminado cuando se inicia el contenedor. Este script se utiliza para iniciar los servicios necesarios de Hadoop, como SSH, NameNode, DataNode, ResourceManager, etc.

In [ ]:
WORKDIR ${HADOOP_HOME}

RUN echo "$LOG_TAG Iniciando servicios"
CMD ["/opt/bd/start-services.sh"]

### **Config-files/`hdfs-site.xml`**

El archivo **hdfs-site-namenode.xml** en Hadoop contiene configuraciones específicas para el NameNode del sistema de archivos distribuido de Hadoop (`HDFS`). Cada propiedad en este archivo configura un aspecto diferente del funcionamiento del NameNode. Vamos a desglosar y explicar los componentes proporcionados.

**Estructura General**

El archivo está envuelto en una etiqueta `<configuration>`, dentro de la cual hay varias etiquetas `<property>`. Cada `<property>` define una propiedad de configuración con sus respectivas etiquetas `<name>`, `<value>`, y opcionalmente `<final>`.

**Componentes del `hdfs-site.xml`**

**1. Configuración de `dfs.datanode.data.dir`**

```xml
                                    <property>
                                        <name>dfs.datanode.data.dir</name>
                                        <value>/opt/bd/hdfs/datanode</value>
                                        <description>DataNode directory</description>
                                    </property>
```
**`<name>dfs.datanode.data.dir</name>`**: Esta propiedad define la ubicación en el sistema de archivos local donde el DataNode debe almacenar los metadatos. **dfs.datanode.data.dir** es el nombre de la propiedad.

**`<value>/opt/bd/hdfs/datanode</value>`**: El valor de esta propiedad es **/opt/bd/hdfs/datanode**, indicando que los metadatos del DataNode se almacenarán en el directorio **/opt/bd/hdfs/datanode** en el sistema de archivos local. Se puede configurar una lista separada por comas de directorios para replicar los metadatos en múltiples ubicaciones, proporcionando redundancia.

### **Config-files/`mapred-site.xml`**

**1. Configuración de `mapreduce.framework.name`**

```xml
                    <property>
                    <!-- El runtime framework para ejecutar MapReduce jobs. Puede ser local, classic o yarn.
                        Por defecto es "local" -->
                        <name>mapreduce.framework.name</name>
                        <value>yarn</value>
                    </property>
```
**`<name>mapreduce.framework.name</name>`**: especifica el framework de tiempo de ejecución (runtime framework) para los MapReduce jobs. Aquí, la propiedad **mapreduce.framework.name** se establece en **yarn**, lo que indica que se utilizará el framework YARN para ejecutar trabajos MapReduce en el clúster.

**`<value>yarn</value>`**: indica que se utilizará el framework YARN para ejecutar trabajos MapReduce en el clúster.

___

**2. Configuración de `mapreduce.application.classpath`**

```xml
            <property>
            <!-- CLASSPATH para aplicaciones MR. Una lista separada por comas de entradas CLASSPATH. 
                Si mapreduce.application.framework está establecido entonces esto debe especificar 
                el classpath apropiado para ese archivo, y el nombre del archivo debe estar presente 
                en el classpath. Si mapreduce.app-submission.cross-platform es falso, se utilizará la 
                sintaxis de expansión vairable del entorno específico de la plataforma para construir 
                las entradas CLASSPATH por defecto. Para Linux: $HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*, 
                $HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*. Para Windows: %HADOOP_MAPRED_HOME%/share/hadoop/mapreduce/*, 
                %HADOOP_MAPRED_HOME%/share/hadoop/mapreduce/lib/*. Si mapreduce.app-submission.cross-platform 
                es true, se utilizará el CLASSPATH por defecto agnóstico de plataforma para aplicaciones 
                MR: {{HADOOP_MAPRED_HOME}}/share/hadoop/mapreduce/*, {{HADOOP_MAPRED_HOME}}/share/hadoop/mapreduce/lib/* 
                El marcador de expansión de parámetros será sustituido por NodeManager en el lanzamiento del contenedor 
                en función del SO subyacente según corresponda. -->
                <name>mapreduce.application.classpath</name>
                <value>/opt/bd/hadoop/share/hadoop/mapreduce/*</value>
            </property>   
```
**`<name>mapreduce.application.classpath</name>`**: especifica el CLASSPATH para las aplicaciones MapReduce. El CLASSPATH es una lista de ubicaciones de directorios y archivos JAR que se utilizan para buscar clases Java durante la ejecución de las aplicaciones MapReduce.

**`<value>/opt/bd/hadoop/share/hadoop/mapreduce/*</value>`**:  indica que se incluirán todos los archivos en el directorio /opt/bd/hadoop/share/hadoop/mapreduce/ en el CLASSPATH para las aplicaciones MapReduce.

La ruta `${HADOOP_HOME}/share/hadoop/mapreduce/*` contiene archivos de clase y recursos necesarios para ejecutar tareas MapReduce en un clúster Hadoop.

- `${HADOOP_HOME}` se refiere al directorio de instalación de Hadoop.
- `share/hadoop/mapreduce` es la ubicación dentro de la instalación de Hadoop donde se encuentran los archivos relacionados con MapReduce.

La expresión `*` al final de la ruta indica que se están incluyendo todos los archivos y subdirectorios dentro de la carpeta `mapreduce`.

Estos archivos suelen incluir archivos JAR de bibliotecas, configuraciones específicas de MapReduce, y otros recursos necesarios para ejecutar tareas de MapReduce en un clúster Hadoop. Por ejemplo, pueden contener clases de MapReduce, archivos de configuración como `mapred-site.xml`, `core-site.xml`, etc., y otros archivos necesarios para el funcionamiento de tareas de MapReduce.

El contenido de esta ruta se agrega al classpath de las tareas de MapReduce cuando se ejecutan en el clúster Hadoop, lo que permite que las clases y recursos necesarios estén disponibles durante la ejecución de estas tareas.

Sí, en muchos casos, especificar `${HADOOP_HOME}/share/hadoop/mapreduce/*` como parte del classpath puede ser suficiente. Esto incluirá todos los archivos y subdirectorios dentro de la carpeta `mapreduce`, lo que generalmente abarca tanto `lib` como `lib-examples`, junto con cualquier otro contenido que pueda existir en esa ubicación.

Al especificar `${HADOOP_HOME}/share/hadoop/mapreduce/*`, estás asegurando que cualquier biblioteca o recurso necesario para las tareas MapReduce, incluidos los ejemplos y las demostraciones, esté disponible en el classpath. Esto es útil si estás ejecutando tareas MapReduce estándar o si estás trabajando con los ejemplos proporcionados por Hadoop.

Sin embargo, si tienes requisitos específicos y necesitas controlar exactamente qué recursos están en el classpath, o si necesitas excluir ciertos recursos, entonces puede ser útil especificar carpetas específicas como `lib` o `lib-examples` en lugar de usar un comodín `*`.

En resumen, si simplemente necesitas asegurarte de que todas las bibliotecas y recursos relevantes de MapReduce estén disponibles en el classpath, `${HADOOP_HOME}/share/hadoop/mapreduce/*` debería ser suficiente.

### **Config-files/`yarn-site.xml`**

**1. Configuración de `yarn.resourcemanager.hostname`**

```xml
                                    <property>
                                        <name>yarn.resourcemanager.hostname</name>
                                        <value>hadoop-master</value>
                                    </property>
```
**`<name>yarn.resourcemanager.hostname</name>`**: especifica el nombre de host del ResourceManager de YARN en el clúster. Dado que en el nodo master se encuentran el namenodey el resourcemanager, ambos pueden utilizar el nombre del host que es "hadoop-master".

La propiedad `yarn.resourcemanager.hostname` se utiliza para configurar la dirección del ResourceManager para que los nodos del clúster puedan comunicarse con él correctamente.

**`<value>hadoop-master</value>`**: indica que el nombre de host del ResourceManager de YARN es `hadoop-master`.

___